# Stage 1 — Build the ChromaDB Job Index

This notebook is designed for Google Colab and reuses the job dataset already stored in Google Drive.

**Existing dataset:** `/content/drive/MyDrive/postings.csv`

**ChromaDB:** `/content/drive/MyDrive/chroma_db`

Workflow: `postings.csv` → cleaning → chunking → embeddings → ChromaDB (`tech_jobs`)


In [8]:
# Install project dependencies
!pip install -q pandas numpy tqdm chromadb langchain-text-splitters


In [2]:
# Mount Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DATASET_PATH = Path('/content/drive/MyDrive/postings.csv')
CHROMA_DB_PATH = Path('/content/drive/MyDrive/chroma_db')
COLLECTION_NAME = 'tech_jobs'
MAX_ROWS = 10_000
BATCH_SIZE = 100

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f'Dataset not found at {DATASET_PATH}. Make sure postings.csv is already in My Drive.'
    )

CHROMA_DB_PATH.mkdir(parents=True, exist_ok=True)

print(f'Dataset found: {DATASET_PATH}')
print(f'ChromaDB path: {CHROMA_DB_PATH}')


Mounted at /content/drive
Dataset found: /content/drive/MyDrive/postings.csv
ChromaDB path: /content/drive/MyDrive/chroma_db


In [3]:
# Load the existing dataset from Google Drive
import pandas as pd

df = pd.read_csv(DATASET_PATH).head(MAX_ROWS)
print(f'Loaded {len(df):,} job postings.')
print(f'Columns: {list(df.columns)}')


Loaded 10,000 job postings.
Columns: ['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']


In [4]:
# Text cleaning and chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

def clean_text(text) -> str:
    if pd.isna(text):
        return ''
    return str(text).replace('\n', ' ').strip()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,
    separators=['\n\n', '\n', '. ', ' ', '']
)


In [5]:
# Initialize ChromaDB and reuse the existing collection when available
import chromadb
from chromadb.utils import embedding_functions

embedding_fn = embedding_functions.DefaultEmbeddingFunction()
client = chromadb.PersistentClient(path=str(CHROMA_DB_PATH))

try:
    collection = client.get_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_fn
    )
    existing_count = collection.count()
    print(f'Existing collection found: {COLLECTION_NAME} ({existing_count:,} chunks)')
except Exception:
    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_fn,
        metadata={'hnsw:space': 'cosine'}
    )
    existing_count = collection.count()
    print(f'Created collection: {COLLECTION_NAME}')


Existing collection found: tech_jobs (10,000 chunks)


In [6]:
# Set this to True only when you intentionally want to rebuild the index
REBUILD_INDEX = False

if REBUILD_INDEX:
    try:
        client.delete_collection(name=COLLECTION_NAME)
    except Exception:
        pass

    collection = client.get_or_create_collection(
        name=COLLECTION_NAME,
        embedding_function=embedding_fn,
        metadata={'hnsw:space': 'cosine'}
    )
    print('Existing collection deleted. A fresh index will be built.')
else:
    if collection.count() > 0:
        print('Existing ChromaDB index will be reused. No re-indexing is needed.')


Existing ChromaDB index will be reused. No re-indexing is needed.


In [7]:
# Index the dataset only when the collection is empty or a rebuild was requested
import uuid

def index_jobs(dataframe: pd.DataFrame, batch_size: int = 100) -> int:
    documents_batch = []
    metadatas_batch = []
    ids_batch = []
    total_chunks = 0

    for idx, row in dataframe.iterrows():
        job_title = row.get('title', 'Unknown Title')
        company = row.get('company', 'Unknown Company')
        description = clean_text(row.get('description', ''))

        if not description:
            continue

        parent_id = f'job_{uuid.uuid4().hex[:8]}'
        chunks = text_splitter.split_text(description)

        for chunk_index, chunk in enumerate(chunks):
            documents_batch.append(chunk)
            metadatas_batch.append({
                'parent_id': parent_id,
                'job_title': str(job_title) if not pd.isna(job_title) else 'Unknown Title',
                'company': str(company) if not pd.isna(company) else 'Unknown Company',
                'chunk_index': chunk_index,
                'total_chunks': len(chunks)
            })
            ids_batch.append(f'{parent_id}_chunk_{chunk_index}')

            if len(documents_batch) >= batch_size:
                collection.add(
                    documents=documents_batch,
                    metadatas=metadatas_batch,
                    ids=ids_batch
                )
                total_chunks += len(documents_batch)
                documents_batch.clear()
                metadatas_batch.clear()
                ids_batch.clear()

        if (idx + 1) % 500 == 0:
            print(f'Processed {idx + 1:,}/{len(dataframe):,} rows...')

    if documents_batch:
        collection.add(
            documents=documents_batch,
            metadatas=metadatas_batch,
            ids=ids_batch
        )
        total_chunks += len(documents_batch)

    return total_chunks

if REBUILD_INDEX or collection.count() == 0:
    total_chunks = index_jobs(df, batch_size=BATCH_SIZE)
    print(f'\nDone. Added {total_chunks:,} chunks.')
else:
    print('\nSkipped indexing because a ChromaDB index already exists.')

print(f'Collection size: {collection.count():,}')
print(f'ChromaDB path: {CHROMA_DB_PATH}')



Skipped indexing because a ChromaDB index already exists.
Collection size: 10,000
ChromaDB path: /content/drive/MyDrive/chroma_db


## Next step

Open **Stage 2 — Semantic Search + RAG Career Advisor**. It can reuse the same ChromaDB collection: `tech_jobs`.
